In [3]:
import csv
import json
import math
from pathlib import Path

In [4]:

# Input folder and output page
folder = Path.cwd()
data_folder = folder / "csv_data"
output_file = folder / "index.html"


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def get_points(rows, column, used_column=None):
    points = []

    for row in rows:
        if used_column and row.get(used_column) != "1":
            continue

        try:
            x = float(row["x_nm"])
            y = float(row[column])
        except (KeyError, TypeError, ValueError):
            continue

        if math.isfinite(x) and math.isfinite(y):
            points.append([x, y])

    # Keep measured points in their original order.
    # Sort the fitted points so the lines follow X.
    if used_column:
        points.sort(key=lambda point: point[0])

    return points


def curve_key(row):
    return (
        row.get("filename", ""),
        row.get("curve_number", ""),
        row.get("position_index", ""),
    )


datasets = []

curve_folders = sorted({
    path.parent
    for path in data_folder.rglob("AFM_fit_points_curve_*.csv")
})

for curve_folder in curve_folders:
    summary_file = curve_folder / "AFM_jython_summary.csv"

    if not summary_file.exists():
        raise FileNotFoundError(
            f"Missing summary CSV in {curve_folder}"
        )

    summary = {
        curve_key(row): row
        for row in read_csv(summary_file)
    }

    curves = []

    for path in sorted(
        curve_folder.glob("AFM_fit_points_curve_*.csv")
    ):
        rows = read_csv(path)

        if not rows:
            print("Skipped empty file:", path.name)
            continue

        first = rows[0]
        values = summary.get(curve_key(first), {})

        if not values:
            print("No matching summary row:", path.name)

        curves.append({
            "number": first["curve_number"],
            "position": first["position_index"],
            "filename": first["filename"],
            "measured": get_points(rows, "measured_y_nN"),
            "linear": get_points(
                rows,
                "linear_fit_y_nN",
                "linear_fit_used",
            ),
            "elasticity": get_points(
                rows,
                "elasticity_fit_y_nN",
                "elasticity_fit_used",
            ),
            "values": values,
        })

    curves.sort(
        key=lambda curve: (
            int(curve["number"]),
            int(curve["position"]),
        )
    )

    if curves:
        datasets.append({
            "name": curve_folder.relative_to(data_folder).as_posix(),
            "curves": curves,
        })

if not datasets:
    raise SystemExit(
        "No curves found. Put your CSV files inside csv_data."
    )


html = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>AFM CSV Dashboard</title>

    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: #f4f5f7;
            color: #202124;
        }

        header {
            padding: 20px 28px;
            background: white;
            border-bottom: 1px solid #ddd;
        }

        h1 {
            margin: 0;
            font-size: 24px;
        }

        header p {
            margin-bottom: 0;
            color: #666;
        }

        .controls, .card {
            background: white;
            padding: 20px;
            border-radius: 8px;
        }

        .controls {
            margin: 20px;
            display: flex;
            align-items: center;
            flex-wrap: wrap;
            gap: 12px;
        }

        select, button {
            padding: 9px 12px;
            font-size: 14px;
        }

        select {
            min-width: 200px;
            max-width: 100%;
        }

        button {
            cursor: pointer;
        }

        .layout {
            display: grid;
            grid-template-columns: minmax(0, 1fr) 360px;
            gap: 20px;
            margin: 20px;
        }

        .card {
            min-width: 0;
        }

        #plot {
            height: 600px;
        }

        h2 {
            margin-top: 0;
            font-size: 20px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }

        td {
            padding: 10px 4px;
            border-bottom: 1px solid #eee;
        }

        td:nth-child(2) {
            text-align: right;
            font-weight: bold;
        }

        td:last-child {
            color: #666;
            padding-left: 10px;
            white-space: nowrap;
        }

        #filename, #status {
            font-size: 13px;
            color: #666;
            line-height: 1.5;
            overflow-wrap: anywhere;
        }

        @media (max-width: 900px) {
            .layout {
                grid-template-columns: 1fr;
            }

            #plot {
                height: 450px;
            }
        }
    </style>
</head>

<body>
    <header>
        <h1>AFM CSV Dashboard</h1>
        <p>Force curves and saved fit results</p>
    </header>

    <div class="controls">
        <label for="folder">Folder</label>
        <select id="folder"></select>

        <label for="curve">Curve</label>
        <select id="curve"></select>

        <button id="previous">Previous</button>
        <button id="next">Next</button>
    </div>

    <div class="layout">
        <div class="card">
            <div id="plot"></div>
        </div>

        <div class="card">
            <h2>Curve values</h2>
            <table>
                <tbody id="values"></tbody>
            </table>

            <p id="status"></p>
            
        </div>
    </div>

    <script>
        const datasets = __DATA__;

        const folderSelect = document.getElementById("folder");
        const curveSelect = document.getElementById("curve");
        const valuesTable = document.getElementById("values");
        const status = document.getElementById("status");

        const fields = [
            ["Linear slope", "linear_slope_bruker_N_per_m", "N/m"],
            ["Bacterial spring constant", "bacterial_spring_constant_N_per_m", "N/m"],
            ["Bacterial spring constant", "bacterial_spring_constant_nN_per_um", "nN/µm"],
            ["Linear RMSD custom", "linear_rmsd_custom_pN", "pN"],
            ["Linear X minimum", "linear_x_min_used_nm", "nm"],
            ["Linear X maximum", "linear_x_max_used_nm", "nm"],
            ["Linear fit points", "linear_T", ""],
            ["Young's modulus", "youngs_modulus_MPa", "MPa"],
            ["JPK Elasticity RMS", "elasticity_rms_bruker_pN", "pN"], 
            ["Elasticity RMSD custom", "elasticity_rmsd_custom_pN", "pN"],
            ["Elasticity X minimum", "elasticity_x_min_used_nm", "nm"],
            ["Elasticity X maximum", "elasticity_x_max_used_nm", "nm"]
            ["Elasticity fit points", "elasticity_T", ""],
            ["Elasticity fit points til contact", "elasticity_contact_T", ""],
        ];

        function formatNumber(value) {
            if (value == null || String(value).trim() === "") {
                return "N/A";
            }

            const number = Number(value);

            return Number.isFinite(number)
                ? String(Number(number.toPrecision(6)))
                : "N/A";
        }

        function makeTrace(points, name, color, mode) {
            return {
                x: points.map(point => point[0]),
                y: points.map(point => point[1]),
                type: "scatter",
                mode: mode,
                name: name,
                marker: { color: color, size: 3 },
                line: { color: color, width: 3 }
            };
        }

        function showCurve() {
            const dataset = datasets[Number(folderSelect.value)];
            const index = Number(curveSelect.value);
            const curve = dataset.curves[index];

            const traces = [
                {
                    x: curve.measured.map(point => point[0]),
                    y: curve.measured.map(point => point[1]),
                    type: "scatter",
                    mode: "lines+markers",
                    name: "Measured",
                    marker: {
                        color: "#58758a",
                        size: 4
                    },
                    line: {
                        color: "#58758a",
                        width: 1.5
                    }
                }
            ];

            if (curve.linear.length) {
                traces.push(
                    makeTrace(
                        curve.linear, "Linear fit", "#2ca02c", "lines"
                    )
                );
            }

            if (curve.elasticity.length) {
                traces.push(
                    makeTrace(
                        curve.elasticity, "Elasticity fit", "#d62728", "lines"
                    )
                );
            }

            Plotly.react("plot", traces, {
                title: `${dataset.name} | Curve ${curve.number}`,
                xaxis: { title: "X (nm)", autorange: true },
                yaxis: { title: "Force (nN)", autorange: true },
                legend: { orientation: "h", y: -0.2 },
                margin: { l: 70, r: 25, t: 60, b: 90 },
                hovermode: "closest"
            }, {
                responsive: true,
                displaylogo: false,
                toImageButtonOptions: {
                    filename: `AFM_curve_${curve.number}`,
                    scale: 2
                }
            });

            valuesTable.replaceChildren();

            for (const [label, column, unit] of fields) {
                const row = document.createElement("tr");

                for (const text of [
                    label,
                    formatNumber(curve.values[column]),
                    unit
                ]) {
                    const cell = document.createElement("td");
                    cell.textContent = text;
                    row.appendChild(cell);
                }

                valuesTable.appendChild(row);
            }

            status.textContent =
                `Curve ${index + 1} of ${dataset.curves.length}. ` +
                `Position index: ${curve.position}.`;

            if (Object.keys(curve.values).length === 0) {
                status.textContent += " No matching summary row.";
            }

            document.getElementById("filename").textContent =
                curve.filename;

            document.getElementById("previous").disabled = index === 0;
            document.getElementById("next").disabled =
                index === dataset.curves.length - 1;
        }

        function showFolder() {
            const dataset = datasets[Number(folderSelect.value)];
            curveSelect.replaceChildren();

            dataset.curves.forEach((curve, index) => {
                curveSelect.add(new Option(
                    `Curve ${curve.number} | Position ${curve.position}`,
                    index
                ));
            });

            showCurve();
        }

        datasets.forEach((dataset, index) => {
            folderSelect.add(new Option(dataset.name, index));
        });

        folderSelect.onchange = showFolder;
        curveSelect.onchange = showCurve;

        document.getElementById("previous").onclick = () => {
            if (curveSelect.selectedIndex > 0) {
                curveSelect.selectedIndex -= 1;
                showCurve();
            }
        };

        document.getElementById("next").onclick = () => {
            if (curveSelect.selectedIndex < curveSelect.length - 1) {
                curveSelect.selectedIndex += 1;
                showCurve();
            }
        };

        if (window.Plotly) {
            showFolder();
        } else {
            status.textContent =
                "Could not load the plotting library. Check your connection and reload.";
        }
    </script>
</body>
</html>
"""

# Embed the CSV data in the page.
# Escape "<" so filenames cannot close the script element.
data_json = json.dumps(
    datasets,
    ensure_ascii=True,
    allow_nan=False,
).replace("<", "\\u003c")

output_file.write_text(
    html.replace("__DATA__", data_json),
    encoding="utf-8",
)

print("Created:", output_file)
print("Folders:", len(datasets))
print("Curves:", sum(len(item["curves"]) for item in datasets))

Created: /workspaces/AFM-plots_Jython/index.html
Folders: 2
Curves: 31
